# jsteer hello-world: word steering

Fit the model's full Jacobian once (`scripts/fit.py --model ...`, cached to
`artifacts/<model-slug>.jac`), then any word vector is an instant CPU matvec:

```
v_l = unit( J_l^T @ w )
```

`w` is a cotangent (a direction at the output: here the mean unembedding row of
the words you want more or less of). `J_l^T @ w` is the pullback of `w` -- the
standard autodiff name for J-transpose applied to a cotangent -- landing the
concept as a residual-stream direction. This is the verified extraction method
(see the README evidence section).

We fit and generate through the model's chat template with thinking on, so
`show_steer` can show, per strength C, the j-space readout, the `<think>` trace,
and the answer. Runtime is steering-lite: `with v(model, C=...): generate(...)`.

In [ ]:
# demo notebook authored by Claude
import sys
from loguru import logger

logger.remove()  # show_steer prints through loguru; route it to the cell output
logger.add(sys.stdout, format="{message}")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian, show_steer

MODEL = "Qwen/Qwen3.5-4B"  # 4B-class: demo material. 0.6B degenerates too easily.
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

## Fit or load the Jacobian

The expensive step (1 forward + ~d_model/dim_batch backwards per prompt) runs
once and caches to `config.cache_path(MODEL)`. `fit_cached` builds it on first
run for any model and loads it afterwards, so reruns are cheap. Prompts are
jlens's WikiText wrapped in the chat template, closer to the distribution we
steer in than raw documents. SHOULD: the repr shows the model's `d_model` and a
source-layer band (the 0.3-0.9 fraction of depth).

In [ ]:
# fit-or-load: builds the cache on first run for ANY model, loads it after.
# chat_corpus wraps jlens's WikiText in the chat template, closer to the
# distribution we steer in (chat + <think>) than raw documents. The lambda means
# WikiText is only built on a cache MISS. dim_batch=4 fits a 4B on a 24GB 3090;
# checkpoint_path makes a multi-hour 4B fit resumable if it dies.
sys.path.insert(0, "..")  # repo root for config.py
import config

jac = Jacobian.fit_cached(model, tok, lambda: config.chat_corpus(tok, 128),
                          config.cache_path(MODEL), layers=(0.3, 0.9), dim_batch=4,
                          checkpoint_path=str(config.cache_path(MODEL, "ckpt")))
jac

## Extract a word vector

`word_vector` pulls the words' unembedding direction back through the cached
Jacobian: no gradients, no forward passes, just a matvec per layer.
+C makes the model say these words more, -C less.

In [ ]:
# Verified method: pull the words' unembedding direction back through J.
# +C makes the model say/lean-toward these words, -C away. Instant CPU matvec.
v = jac.word_vector(model, tok, ["happy", "joy"])

## Pick a coefficient: the coherence/strength tradeoff

The raw coefficient is model-dependent, so sweep it. A moderate +C moves the
tone while the text and the `<think>` reasoning stay fluent; too large a C
overwhelms the residual stream and the output degenerates into token spam.
SHOULD: C=0 is the baseline; a moderate +C reads happier and stays coherent;
large |C| degenerates. Watch the j-space row: the concept's tokens should climb
with +C.

In [ ]:
# One identical block per strength C (Tufte small-multiples): the j-space top-k
# at the top layer (what the steered residual "thinks"), the <think> reasoning,
# then the answer. All under steering, through the chat template + the model's
# own sampling. Read down the column against the C=0 baseline.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(-6, 0, 6, 12))

## Steer across prompts

A moderate C keeps the model fluent while moving the tone. The same vector on a
few different user questions, baseline vs +C.

In [ ]:
# Same vector, a few different user prompts, at the baseline vs one +C.
for msg in ("What did you think of the meeting this afternoon?",
            "Give me your honest impression of the new apartment.",
            "How was your commute today?"):
    show_steer(jac, model, tok, v, msg, Cs=(0, 6))

## Negative steering

The same vector with a negative coefficient suppresses the concept.
SHOULD: less positive affect than C=0, still english (strongly negative C
degenerates the same way strongly positive does).

In [ ]:
# Negative steering: the same vector at -C suppresses the concept.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(0, -6))

## Bonus: lens readout

Only the full-Jacobian cache gives you this: transport any layer's residual to
the final basis with `J_l` and decode it, a linear-approximation readout of what
that layer's residual points to in vocab space. SHOULD: on a factual prompt,
deeper layers resolve from a generic slot (e.g. " city") toward the specific
answer (e.g. " Paris"). ELSE layer indexing is off.

In [ ]:
# Only the full-Jacobian cache gives this: transport a layer's residual to the
# final basis and decode it, i.e. "what does the model think at layer l".
# Pick a low / mid / high layer from the fitted band.
lo, mid, hi = jac.layers[0], jac.layers[len(jac.layers) // 2], jac.layers[-1]
for layer in (lo, mid, hi):
    top = jac.lens_topk(model, tok, "The Eiffel Tower is located in the city of", layer=layer, k=6)
    print(f"layer {layer}: {[t for t, _ in top]}")

## Extract once, steer forever

The steering vector is a plain steering-lite `Vector` (safetensors on disk),
so you can save it and reuse it without the Jacobian cache or jsteer at all.

In [ ]:
# The vector is a plain steering-lite Vector: save it, reuse it with no Jacobian
# cache and no jsteer at apply time (only the chat template + steering-lite).
from steering_lite import Vector

v.save("../artifacts/happy_joy.safetensors")
v2 = Vector.load("../artifacts/happy_joy.safetensors")

msg = [{"role": "user", "content": "Describe how your week has been going."}]
prompt = tok.apply_chat_template(msg, add_generation_prompt=True, tokenize=False, enable_thinking=True)
enc = tok(prompt, return_tensors="pt").to(model.device)
with v2(model, C=6):
    out = model.generate(**enc, max_new_tokens=200, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True))